<a href="https://colab.research.google.com/github/MysaniThiya/Agentra-RajarataUniversityOfSriLanka/blob/feature%2Femail-analysis/email_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git --version

git version 2.43.0


In [2]:
!git clone https://github.com/MysaniThiya/Agentra-RajarataUniversityOfSriLanka.git


Cloning into 'Agentra-RajarataUniversityOfSriLanka'...
remote: Enumerating objects: 93, done.
remote: Counting objects: 100% (93/93), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 93 (delta 37), reused 44 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (93/93), 95.17 KiB | 4.53 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [3]:
%cd Agentra-RajarataUniversityOfSriLanka


/content/Agentra-RajarataUniversityOfSriLanka


In [4]:
!ls


Agentra-RajarataUniversityOfSriLanka  project		README.md
docs				      project-demo.mp4


In [5]:
!git fetch origin

In [6]:
!git branch -a

* main
  remotes/origin/Fasna
  remotes/origin/HEAD -> origin/main
  remotes/origin/feature/email-analysis
  remotes/origin/main
  remotes/origin/url-analysis-agent


In [7]:
!git checkout -b feature/email-analysis origin/feature/email-analysis

branch 'feature/email-analysis' set up to track 'origin/feature/email-analysis'.
Switched to a new branch 'feature/email-analysis'


In [8]:
!git branch


* feature/email-analysis
  main


In [9]:
import os

os.makedirs("datasets", exist_ok=True)
os.makedirs("notebooks", exist_ok=True)
os.makedirs("models", exist_ok=True)

In [10]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib datasets

In [11]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [12]:
from datasets import load_dataset

dataset_url = (
    "https://huggingface.co/datasets/"
    "locuoco/the-biggest-spam-ham-phish-email-dataset-300000/"
    "resolve/main/df.csv"
)

# Load the CSV as a streaming dataset
email_stream = load_dataset(
    "csv",
    data_files=dataset_url,
    split="train",
    streaming=True
)

# Shuffle and select 60,000 records for the first experiment
email_stream = email_stream.shuffle(
    seed=42,
    buffer_size=10000
)

records = list(email_stream.take(60000))

# Convert records into a DataFrame
df = pd.DataFrame(records)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (60000, 2)


,label,text
0,0,daren i know it ' s first of the month but the...
1,0,jodi chris can set up a conference call with m...
2,0,the recruiting season has officially begun the...
3,0,attached is a memo summarizing policy changes ...
4,0,please let me know if you are available for a ...


In [13]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nLabel distribution:")
print(df["label"].value_counts().sort_index())

print("\nEmail examples:")
display(df[["text", "label"]].head(10))

Column names:
['label', 'text']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   60000 non-null  int64 
 1   text    60000 non-null  object
dtypes: int64(1), object(1)
memory usage: 937.6+ KB

Label distribution:
label
0    32329
2    27671
Name: count, dtype: int64

Email examples:


,text,label
0,daren i know it ' s first of the month but the...,0
1,jodi chris can set up a conference call with m...,0
2,the recruiting season has officially begun the...,0
3,attached is a memo summarizing policy changes ...,0
4,please let me know if you are available for a ...,0
5,louise effective escapenumber escapenumber esc...,0
6,new power plant added to pnm system pnm public...,0
7,iv friends i spoken with most of you over the ...,0
8,sheila changes to the manual list for north am...,0
9,i also have an opportunity for a manager top c...,0


In [14]:
label_mapping = {
    0: "Ham / Safe",
    1: "Phishing / Malicious",
    2: "Spam / Provisional Suspicious"
}

print("Original label distribution:")

for label, count in df["label"].value_counts().sort_index().items():
    print(label_mapping.get(int(label), "Unknown"), ":", count)

Original label distribution:
Ham / Safe : 32329
Spam / Provisional Suspicious : 27671


In [15]:
from datasets import load_dataset
from collections import Counter

dataset_url = (
    "https://huggingface.co/datasets/"
    "locuoco/the-biggest-spam-ham-phish-email-dataset-300000/"
    "resolve/main/df.csv"
)

# Load the complete dataset as a stream
full_stream = load_dataset(
    "csv",
    data_files=dataset_url,
    split="train",
    streaming=True
)

# Count every original label
label_counts = Counter()

for row in full_stream:
    label_counts[row["label"]] += 1

print("Complete dataset label distribution:")

for label, count in sorted(label_counts.items()):
    print(f"Label {label}: {count:,} emails")

Complete dataset label distribution:
Label 0: 168,455 emails
Label 1: 42,845 emails
Label 2: 154,148 emails
